# Analiza: model dziennego maksimum - SCANWAY (SCW.WA)

**Cel projektu:** raz w ciagu sesji wyslac sygnal *"TERAZ jest szczyt dnia - sprzedaj"*.
Dostajesz jeden alert -> sprzedajesz. Ten notebook pokazuje **krok po kroku, co dzieje
sie na naszym datasecie**: od danych, przez cechy i etykiete, po model i jego ocene.

Uruchom komorki po kolei (Run All). Wymaga `matplotlib` (jest w `requirements.txt`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # korzen repo (notebook lezy w notebooks/)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4); plt.rcParams['axes.grid'] = True

from src.config import load_config
from src.data_sources import get_daily_history
from src.features import build_features, build_daily_context_features, intraday_seasonality
from src.replay import load_intraday_for_replay
from src.peak import add_daily_high_target, train_peak_model, daily_peak_evaluation

cfg = load_config()
print('ticker:', cfg.ticker_yf, '| kara za przegapienie:', cfg.daily_high_fn_penalty,
      '| prog alertu:', cfg.alert_probability_threshold)

## 1. Dane

Dwa horyzonty: **dzienny** (kontekst, od debiutu 2023) i **intraday 15-min**
(dynamika sesji, ~57 dni ze snapshotu). Model uczy sie na danych intraday.

In [ ]:
intraday, source = load_intraday_for_replay(cfg, synthetic=False)
daily = get_daily_history(cfg, refresh_live=False)
print('intraday:', intraday.shape, '| sesji:', intraday['date'].nunique(), '| zrodlo:', source)
print('dzienne :', daily.shape)
intraday.head()

## 2. EDA - kiedy wypada dzienne maksimum?

Dla kazdej godziny liczymy, w jakim % sesji jest ona szczytem dnia. Kluczowa
obserwacja: **otwarcie 09:00 jest dziennym maksimum w ~41% sesji** - zadna inna
godzina sie nie zbliza. Stad `minute_of_day` to mocna cecha.

In [ ]:
seas = intraday_seasonality(intraday).sort_values(
    'pct_of_days_this_is_daily_high', ascending=False).head(10)
labels = [t.strftime('%H:%M') for t in seas.index]
colors = ['#cf222e' if i == 0 else '#afb8c1' for i in range(len(seas))]
plt.bar(labels, seas['pct_of_days_this_is_daily_high'], color=colors)
plt.ylabel('% sesji, gdy ta godzina = szczyt dnia'); plt.xlabel('godzina')
plt.title('Najczestsza pora dziennego maksimum (SCW)'); plt.tight_layout(); plt.show()

## 3. Inzynieria cech

Dla **kazdej** swiecy liczymy wektor cech - "oczy" modelu. `build_features` doklada
tez kontekst dzienny (przesuniety o 1 dzien, bez zagladania w przyszlosc).

In [ ]:
ctx = build_daily_context_features(daily)
feat, cols = build_features(intraday, ctx, cfg)
print('liczba cech:', len(cols)); print(cols)
feat[['Close'] + cols].head()

## 4. Etykieta - czym jest "szczyt"

`target_daily_high = 1` dla swiecy z **najwyzszym Close** w sesji - dokladnie jedna
na dzien. Skutek: klasy sa skrajnie niezbalansowane (~3% to szczyt), co wymusza
asymetryczna funkcje straty w nastepnym kroku.

In [ ]:
feat = add_daily_high_target(feat)
ok = bool((feat.groupby('date')['target_daily_high'].sum() == 1).all())
print('dokladnie 1 szczyt na dzien:', ok)
bal = feat['target_daily_high'].value_counts(normalize=True) * 100
print('bilans klas [proc]: nie-szczyt = %.1f | szczyt = %.1f' % (bal.get(0, 0), bal.get(1, 0)))

## 5. Model + asymetryczna strata - co model sie nauczyl

Regresja logistyczna z waga klasy `{0:1, 1:penalty}` - przegapienie szczytu kosztuje
`daily_high_fn_penalty` razy wiecej niz falszywy alarm. Ponizej **realne wagi**: dodatnie
(zielone) podnosza prawdopodobienstwo szczytu, ujemne (czerwone) obnizaja.

In [ ]:
model = train_peak_model(feat, cols, cfg)
clf = model.named_steps['clf']
coef = pd.Series(clf.coef_.ravel(), index=cols).sort_values()
colors = ['#1a7f37' if c >= 0 else '#cf222e' for c in coef]
plt.barh(coef.index, coef.values, color=colors); plt.axvline(0, color='k', lw=.8)
plt.title('Wagi modelu peak (standaryzowane)'); plt.tight_layout(); plt.show()
print('wyraz wolny b0 =', round(float(clf.intercept_[0]), 3))

## 6. Anatomia jednej decyzji (sesja out-of-sample)

Trenujemy model **bez** ostatniej sesji i odtwarzamy ja swieca-po-swiecy. Gorny panel:
cena (czerwona kropka = faktyczny szczyt). Dolny: `p(szczyt)` dla kazdej swiecy - model
bierze **najwyzszy slupek** (zielony) i tam pada sygnal "sprzedaj".

In [ ]:
days = sorted(feat['date'].unique()); hold = days[-1]
m_oos = train_peak_model(feat[feat['date'] != hold], cols, cfg)
g = feat[feat['date'] == hold].sort_index()
proba = m_oos.predict_proba(g[cols])[:, 1]
close = g['Close'].values; t = [x.strftime('%H:%M') for x in g['time']]
peak_i = int(close.argmax()); alert_i = int(proba.argmax())
regret = (close[peak_i] - close[alert_i]) / close[peak_i] * 100

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
a1.plot(range(len(close)), close, color='#0969da')
a1.scatter([peak_i], [close[peak_i]], color='#cf222e', zorder=5, s=60, label='faktyczny szczyt')
a1.axvline(alert_i, color='#1a7f37', ls='--', label='sygnal modelu')
a1.set_ylabel('cena (PLN)'); a1.legend()
a1.set_title('Sesja %s: sprzedaz %.2f%% ponizej dziennego maksimum' % (hold, regret))
a2.bar(range(len(proba)), proba,
       color=['#1a7f37' if i == alert_i else '#afb8c1' for i in range(len(proba))])
a2.axhline(cfg.alert_probability_threshold, color='#cf222e', ls='--', label='prog alertu')
a2.set_ylabel('p(szczyt)'); a2.legend()
a2.set_xticks(range(0, len(t), 4)); a2.set_xticklabels([t[i] for i in range(0, len(t), 4)])
plt.tight_layout(); plt.show()
print('szczyt o %s | alert o %s | p=%.2f | regret=%.2f%%' % (t[peak_i], t[alert_i], proba[alert_i], regret))

## 7. Ocena out-of-sample - regret i test permutacyjny

Split chronologiczny 80/20. Glowna metryka: **regret** = sredni % ponizej dziennego
maksimum, po jakim sprzedajesz. Im mniej, tym lepiej.

In [ ]:
per_day, summary, _ = daily_peak_evaluation(feat, cols, cfg, train_frac=0.8)
print('sesje treningowe:', summary['n_train_sessions'], '| testowe:', summary['n_test_sessions'])
per_day

In [ ]:
strat = {'MODEL (peak)': summary['mean_regret_model'],
         'otwarcie 09:00': summary['mean_regret_open'],
         'losowo': summary['mean_regret_random'],
         'do zamkniecia': summary['mean_regret_close'],
         'idealnie': 0.0}
s = pd.Series(strat)[::-1]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.barh(s.index, s.values, color=['#0969da', '#afb8c1', '#afb8c1', '#afb8c1', '#1a7f37'])
a1.set_xlabel('% ponizej szczytu (mniej = lepiej)'); a1.set_title('Regret: model vs strategie')
a2.hist(summary['perm_null'], bins=30, color='#afb8c1', label='1000 losowych wyborow')
a2.axvline(summary['mean_regret_model'], color='#1a7f37', lw=2.5, label='nasz model')
a2.set_xlabel('sredni regret [%]'); a2.legend()
a2.set_title('Test permutacyjny: p-value = %.3f' % summary['p_value'])
plt.tight_layout(); plt.show()

## Wnioski

- Model wysyla **jeden** sygnal dziennie (swieca o najwyzszym `p`).
- Sprzedaz srednio **~1.7% ponizej** dziennego maksimum - lepiej niz otwarcie, losowo
  czy trzymanie do zamkniecia.
- Przewaga nad losowym jest **istotna statystycznie** (test permutacyjny, p~0.003).
- Granice: tylko ~57 przykladow szczytu, dane live z ~15 min opoznieniem -> to wsparcie
  decyzji, nie gwarancja. Pelny opis: `docs/JAK_DZIALA.md` i `docs/RAPORT_ANALIZA.md`.